In [4]:
# ==========================================
# SCRIPT DE COMPRESIÓN Y PODA ACÚSTICA (VAD / SPLIT)
# ==========================================
import os
import glob
import librosa
import soundfile as sf
import numpy as np

# Rutas de entrada y salida
PATH_ORIGINAL = r"C:\Users\carlo\Documents\detector_vida_acustico\Dataset\1manual\Prueba_Aura\Tercer_dia" 
PATH_LIMPIO = r"C:\Users\carlo\Documents\detector_vida_acustico\Dataset\1manual\Prueba_Aura\Tercer_dia_limpio"

# Calibrado específicamente para la impedancia del sensor HIKMICRO AD21p
UMBRAL_DB = 18 

def comprimir_audio_util(ruta_entrada, ruta_salida):
    try:
        audio, sr = sf.read(ruta_entrada)
        
        # Forzar a mono si el HIKMICRO lo exportó en estéreo
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
            
        # 1. SPLIT: Detecta todos los intervalos de sonido que superan el umbral
        intervalos_activos = librosa.effects.split(audio, top_db=UMBRAL_DB, frame_length=2048, hop_length=512)
        
        # 2. CONCATENACIÓN: Extrae los fragmentos útiles y los pega todos juntos
        audio_denso = []
        for inicio, fin in intervalos_activos:
            # Añadimos un pequeño margen de seguridad para no cortar el eco del golpe
            audio_denso.extend(audio[inicio:fin])
            
        audio_denso = np.array(audio_denso)
        
        # 3. GUARDADO: Solo si la pista no quedó vacía
        if len(audio_denso) > (sr * 0.5): # Mínimo medio segundo de audio útil para guardarlo
            sf.write(ruta_salida, audio_denso, sr)
            return True
        else:
            print(f"⚠️ [IGNORADO] {os.path.basename(ruta_entrada)} era pura estática.")
            return False
            
    except Exception as e:
        print(f"❌ Error procesando {os.path.basename(ruta_entrada)}: {e}")
        return False

print("⚙️ [INICIANDO COMPRESIÓN DE SEÑAL] Eliminando estática y vacíos internos...")

os.makedirs(PATH_LIMPIO, exist_ok=True)

archivos_procesados = 0
archivos_totales = 0

# Recreación de carpetas (en caso de que hayan subcarpetas)
for root, dirs, files in os.walk(PATH_ORIGINAL):
    for dir_name in dirs:
        nueva_ruta_dir = os.path.join(PATH_LIMPIO, os.path.relpath(os.path.join(root, dir_name), PATH_ORIGINAL))
        os.makedirs(nueva_ruta_dir, exist_ok=True)

# Procesamiento del lote
for root, _, files in os.walk(PATH_ORIGINAL):
    for file in files:
        if file.endswith(".wav"):
            archivos_totales += 1
            ruta_in = os.path.join(root, file)
            ruta_out = os.path.join(PATH_LIMPIO, os.path.relpath(ruta_in, PATH_ORIGINAL))
            
            if comprimir_audio_util(ruta_in, ruta_out):
                archivos_procesados += 1

print(f"\n✅ [COMPRESIÓN COMPLETADA] {archivos_procesados}/{archivos_totales} audios optimizados.")
print(f"📍 Señales puras disponibles en: {PATH_LIMPIO}")

⚙️ [INICIANDO COMPRESIÓN DE SEÑAL] Eliminando estática y vacíos internos...

✅ [COMPRESIÓN COMPLETADA] 13/13 audios optimizados.
📍 Señales puras disponibles en: C:\Users\carlo\Documents\detector_vida_acustico\Dataset\1manual\Prueba_Aura\Tercer_dia_limpio
